In [0]:
base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/"

In [0]:
%sql 
select * from parquet.`/Volumes/nyc/default/yellowtaxi/*.parquet` limit 10; 

In [0]:
dbutils.fs.ls("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow")

In [0]:
df = spark.read.table("samples.nyctaxi.trips") 
df.count()

In [0]:
df.select("dropoff_zip", "pickup_zip").distinct().count()

In [0]:
df.display()

In [0]:

df = spark.read.parquet("/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2026-01.parquet")


ddl_schema = df.schema.toDDL()

print(ddl_schema)

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType

yellowSchema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True)
])


yellow_taxi_df = (
    spark.read
    .format("parquet")
    .schema(yellowSchema)
    .load(yellow_path)
)

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType

yellowSchema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("ehail_fee", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_type", LongType(), True),
    StructField("trip_type", LongType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True)
])


yellow_taxi_df = (
    spark.read
    .format("parquet")
    .schema(yellowSchema)
    .load(yellow_path)
)



In [0]:
%sql
select * from parquet.`/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2026-01.parquet` limit 10

In [0]:
yellow_taxi_df.select("DOLocationID", "PULocationID").dropDuplicates().count()

In [0]:
pip install h3pandas 


In [0]:
%pip install geopandas 

In [0]:
import geopandas as gpd 

gdf = gpd.read_file("/Volumes/nyc/default/nyczone/taxi_zones/taxi_zones.shp")

gdf['centroid_lat'] = gdf.geometry.centroid.y
gdf['centroid_lon'] = gdf.geometry.centroid.x 



In [0]:
import h3
import geopandas as gpd

# 加载数据
gdf = gpd.read_file("/Volumes/nyc/default/nyczone/taxi_zones/taxi_zones.shp").to_crs("EPSG:4326")

def get_h3_fast_fix(geom, res=8):
    """
    终极方案：直接使用质心点转换。
    避开所有 Polygon 对象和 API 兼容性问题。
    """
    try:
        # 获取几何中心点
        centroid = geom.centroid
        lat, lng = centroid.y, centroid.x
        
        # 针对 H3 4.x 的函数名
        if hasattr(h3, 'latlng_to_cell'):
            return [h3.latlng_to_cell(lat, lng, res)]
        # 针对 H3 3.x 的函数名
        else:
            return [h3.geo_to_h3(lat, lng, res)]
    except:
        return []

gdf['h3_cells'] = gdf['geometry'].apply(lambda x: get_h3_fast_fix(x, 8))

# 统计成功数
print(f"成功填充的区域数: {gdf[gdf['h3_cells'].map(len) > 0].shape[0]} / 263")

In [0]:
print(gdf.columns)

In [0]:
# 1. 使用你查询出的准确列名进行选择和炸开
h3_df_exploded = gdf[['LocationID', 'borough', 'zone', 'h3_cells']].explode('h3_cells')

# 2. 预览数据：确认空间映射是否直观
# 我们可以根据 borough 分组看看每个区的网格分布情况
display(h3_df_exploded.head(15))

# 3. 统计一下每个区域的网格数，验证“合理性”
print("\n每个行政区的网格覆盖统计：")
print(h3_df_exploded.groupby('borough').size().sort_values(ascending=False))

In [0]:
# 看看哪两个区域最广阔
print(h3_df_exploded.groupby('zone').size().sort_values(ascending=False).head(5))

In [0]:
%pip install folium

In [0]:
import h3
import geopandas as gpd
import folium
from shapely.geometry import mapping

# 1. 加载并转换坐标系 (确保列名匹配你的 Index: zone, LocationID, borough)
gdf = gpd.read_file("/Volumes/nyc/default/nyczone/taxi_zones/taxi_zones.shp").to_crs("EPSG:4326")

# 2. 定义 H3 转换函数 (使用你之前调通的质心方案，最稳健)
def get_h3_cell(geom, res=8):
    centroid = geom.centroid
    try:
        # 兼容 H3 4.x 和 3.x
        return h3.latlng_to_cell(centroid.y, centroid.x, res) if hasattr(h3, 'latlng_to_cell') else h3.geo_to_h3(centroid.y, centroid.x, res)
    except:
        return None

# 3. 生成数据
gdf['h3_cells'] = gdf['geometry'].apply(get_h3_cell)
# 这一步定义变量 h3_df_exploded
h3_df_exploded = gdf[['LocationID', 'borough', 'zone', 'h3_cells']].copy()

# 4. 创建地图
m = folium.Map(location=[40.7306, -73.9352], zoom_start=11, tiles='CartoDB positron')

# 5. 抽样可视化 (取 50 个点防止浏览器卡死)
sample_df = h3_df_exploded.sample(50)

for _, row in sample_df.iterrows():
    cell = row['h3_cells']
    if cell:
        # 获取六边形边界
        try:
            boundary = h3.cell_to_boundary(cell)
        except:
            boundary = h3.h3_to_geo_boundary(cell)
            
        # 绘制
        folium.Polygon(
            locations=boundary,
            popup=f"Zone: {row['zone']}<br>ID: {row['LocationID']}",
            color='crimson',
            fill=True,
            fill_opacity=0.4
        ).add_to(m)

# 6. 显示地图
m

In [0]:
print(type(yellow_taxi_df))

In [0]:
print(type(h3_df_exploded))

In [0]:
# 1. 将 Pandas DataFrame 转换为 Spark DataFrame
# 注意：转换时 Spark 会根据 Pandas 的数据自动推断 Schema
spark_h3_df = spark.createDataFrame(h3_df_exploded)

# 2. 执行 Join (现在两边都是 Spark DataFrame 了)
enriched_trips = yellow_taxi_df.join(
    spark_h3_df.select("LocationID", "h3_cells"), 
    on=yellow_taxi_df.PULocationID == spark_h3_df.LocationID, 
    how="inner"
)

# 3. 检查结果
# 注意：请确认字段名是 lpep_... 还是 tpep_...（根据你之前定义的 Schema 应该是 lpep）
enriched_trips.select("tpep_pickup_datetime", "PULocationID", "h3_cells").show(10)

In [0]:
from pyspark.sql.functions import col, count

# 统计每个六边形内的订单总数
h3_counts = (
    enriched_trips
    .groupBy("h3_cells")
    .agg(count("*").alias("trip_count"))
    .orderBy(col("trip_count").desc())
)

h3_counts.show(10)

In [0]:
display(h3_counts)

In [0]:
from pyspark.sql.functions import col, expr, regexp_extract

# 1. 使用 expr 调用 SQL 中的 h3_centeraswkt 函数
# 2. 使用 regexp_extract 提取括号中的经纬度数值
enriched_trips_geo = enriched_trips.withColumn("h3_wkt", expr("h3_centeraswkt(h3_cells)")) \
    .withColumn("longitude", regexp_extract(col("h3_wkt"), r"POINT\((.+) .+\)", 1).cast("double")) \
    .withColumn("latitude", regexp_extract(col("h3_wkt"), r".+ (.+)\)", 1).cast("double"))

# 3. 再次尝试 display
display(enriched_trips_geo)

In [0]:
# 1. 统计每个 H3 格子的订单量
h3_summary = enriched_trips_geo.groupBy("h3_cells", "latitude", "longitude").count()

# 2. 这里的 count 就是我们的“热度”
display(h3_summary)

In [0]:
import plotly.express as px

# 转为 Pandas 方便绘图 (注意：数据量大时先 limit，或者只传聚合后的 h3_summary)
pdf = h3_summary.limit(5000).toPandas()

fig = px.scatter_mapbox(pdf, 
                        lat="latitude", 
                        lon="longitude", 
                        size="count", 
                        color="count",
                        color_continuous_scale=px.colors.cyclical.IceFire, 
                        size_max=15, 
                        zoom=10,
                        mapbox_style="carto-positron")

fig.show()

In [0]:
import plotly.express as px

# 1. 将数据转换为 Pandas (绘图库需要)
# 建议先聚合，并限制数据量（例如 10000 行）以防浏览器卡顿
plot_df = h3_summary.orderBy("count", ascending=False).limit(10000).toPandas()

# 2. 使用密度地图 (Density Mapbox) 模式
# 这种模式会自动根据点的密集程度生成平滑的颜色过渡（真正的热力图）
fig = px.density_mapbox(
    plot_df, 
    lat='latitude', 
    lon='longitude', 
    z='count',          # 权重列：订单越多，颜色越深
    radius=12,          # 热力点半径，可以根据缩放程度调整
    center=dict(lat=40.7128, lon=-74.0060), # 默认中心点：纽约
    zoom=10, 
    mapbox_style="carto-positron",
    title="NYC Yellow Taxi H3 订单密度热力图",
    height=600
)

# 3. 显示图表
fig.show()

In [0]:
# 如果你有 Python 列表想传进去，这样做：
ids_to_exclude = [236, 237, 161]
filter_str = f"PULocationID NOT IN ({','.join(map(str, ids_to_exclude))})"
enriched_trips.filter(filter_str).show()